# TP2 Notebook 02 : traitement avec PySpark

Sekou Camara, B2 Sup de Vinci Rennes

On lit les collections MongoDB avec Spark, on fait des transformations et des actions, on regarde Spark UI et on calcule les indicateurs du projet.

## 1. Préparer PySpark

Spark tourne en mode local avec `local[*]` : il utilise tous les cœurs du Mac. On charge le connecteur MongoDB version 11, celui qui marche avec Spark 4.

In [1]:
import sys
import subprocess
import pyspark
from pyspark.sql import SparkSession, Window, functions as F

print("Python  :", sys.version.split()[0])
print("PySpark :", pyspark.__version__)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])

Python  : 3.12.6
PySpark : 4.0.1
openjdk version "17.0.21.1" 2026-11-17


In [2]:
MONGO_URI = "mongodb://localhost:27017"
DB = "commerce"

spark = (
    SparkSession.builder
    .appName("BigData Commerce Analytics TP2")
    .master("local[*]")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.0.0")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark   :", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/26 11:58:12 WARN Utils: Your hostname, MacBook-Pro-de-Sekou.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
26/11/26 11:58:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/sekou/tp2-bigdata/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/sekou/.ivy2.5.2/cache
The jars for the packages stored in: /Users/sekou/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-850e2219-8cad-45ac-beeb-909c42c85a4a;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;11.0.0 in central


	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
:: resolution report :: resolve 3010ms :: artifacts dl 9ms
	:: modules in use:
	org.mongodb#bson;5.1.4 from central in [default]
	org.mongodb#bson-record-codec;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-core;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-sync;5.1.4 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.13;11.0.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   5   |   1   |   0   |   0   

26/11/26 11:58:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark   : 4.0.1
Spark UI: http://172.20.10.3:4040


On ouvre Spark UI à l'adresse http://localhost:4040 et on la laisse ouverte.

## 2. Lire les données depuis MongoDB

In [3]:
def lire(collection):
    return spark.read.format("mongodb").option("database", DB).option("collection", collection).load()


commandes = lire("raw_commandes")
paiements = lire("raw_payments")
commandes.printSchema()

root
 |-- _id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cart_discounted_total: double (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- cart_total: double (nullable = true)
 |-- category: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- discounted_line_total: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)



In [4]:
paiements.printSchema()

root
 |-- _id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- attempted_at: timestamp (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- is_cashed: boolean (nullable = true)
 |-- nb_lignes: integer (nullable = true)
 |-- payment_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- user_id: integer (nullable = true)



In [5]:
commandes.show(5, truncate=False)
paiements.show(5, truncate=False)

+------------------------+--------------+---------------------+-------+----------+------------------+-------------------+---------------------+-----------------+-------------------+----------+----------+----------------------+--------+----------+-------+--------+
|_id                     |brand         |cart_discounted_total|cart_id|cart_total|category          |discount_percentage|discounted_line_total|line_total       |order_date         |order_id  |product_id|product_title         |quantity|unit_price|user_id|username|
+------------------------+--------------+---------------------+-------+----------+------------------+-------------------+---------------------+-----------------+-------------------+----------+----------+----------------------+--------+----------+-------+--------+
|6b0810c0c2ae0ae53743f7c7|NULL          |11510.81             |1      |13037.88  |tops              |12.13              |105.41               |119.96           |2025-03-02 01:00:00|CART-00001|162       |Blue 

+------------------------+--------+-------------------+-------+---------+---------+----------+-------+-------+
|_id                     |amount  |attempted_at       |cart_id|is_cashed|nb_lignes|payment_id|status |user_id|
+------------------------+--------+-------------------+-------+---------+---------+----------+-------+-------+
|6b0810c0c2ae0ae53743fae7|11510.81|2025-03-02 01:00:00|1      |false    |4        |PAY00001  |REFUSED|1      |
|6b0810c0c2ae0ae53743fae8|125.7   |2025-02-22 01:00:00|2      |true     |2        |PAY00002  |PAID   |2      |
|6b0810c0c2ae0ae53743fae9|1625.77 |2025-04-25 02:00:00|3      |false    |6        |PAY00003  |REFUSED|3      |
|6b0810c0c2ae0ae53743faea|636.81  |2025-09-09 02:00:00|4      |true     |2        |PAY00004  |PAID   |4      |
|6b0810c0c2ae0ae53743faeb|1205.8  |2025-05-03 02:00:00|5      |true     |3        |PAY00005  |PAID   |5      |
+------------------------+--------+-------------------+-------+---------+---------+----------+-------+-------+
o

In [6]:
nb_commandes = commandes.count()
nb_paiements = paiements.count()
print("Lignes raw_commandes :", nb_commandes, "(attendu : 800)")
print("Lignes raw_payments  :", nb_paiements, "(attendu : 208)")

Lignes raw_commandes : 800 (attendu : 800)
Lignes raw_payments  : 208 (attendu : 208)


**Le champ `_id`** : MongoDB ajoute tout seul un identifiant unique à chaque document. Il ne sert à rien pour notre analyse, donc on l'enlève.

In [7]:
commandes = commandes.drop("_id")
paiements = paiements.drop("_id")

## 3. Transformations et actions

In [8]:
lignes = (
    commandes
    .select("cart_id", "user_id", "category", "discounted_line_total")
    .filter(F.col("discounted_line_total") > 0)
    .withColumn(
        "gamme",
        F.when(F.col("discounted_line_total") >= 500, "haute")
        .when(F.col("discounted_line_total") >= 100, "moyenne")
        .otherwise("basse"),
    )
)

par_categorie = (
    lignes
    .groupBy("category")
    .agg(
        F.count("*").alias("nb_lignes"),
        F.countDistinct("cart_id").alias("nb_paniers"),
        F.round(F.sum("discounted_line_total"), 2).alias("montant_total"),
        F.round(F.avg("discounted_line_total"), 2).alias("montant_moyen"),
    )
    .orderBy(F.desc("montant_total"))
)

In [9]:
par_categorie.show(10, truncate=False)
print("Nombre de catégories :", par_categorie.count())
print("Première ligne       :", par_categorie.first())
print("Top 3                :", par_categorie.take(3))
lignes.groupBy("gamme").count().orderBy("gamme").show()

+------------------+---------+----------+-------------+-------------+
|category          |nb_lignes|nb_paniers|montant_total|montant_moyen|
+------------------+---------+----------+-------------+-------------+
|vehicle           |19       |19        |1682064.05   |88529.69     |
|motorcycle        |23       |22        |609460.61    |26498.29     |
|mens-watches      |26       |24        |593063.86    |22810.15     |
|womens-watches    |18       |17        |249189.51    |13843.86     |
|laptops           |16       |16        |85006.24     |5312.89      |
|furniture         |22       |21        |77674.97     |3530.68      |
|smartphones       |60       |52        |73130.95     |1218.85      |
|mobile-accessories|70       |61        |26160.99     |373.73       |
|tablets           |14       |13        |14890.16     |1063.58      |
|mens-shoes        |23       |22        |8278.9       |359.95       |
+------------------+---------+----------+-------------+-------------+
only showing top 10 

Nombre de catégories : 24


Première ligne       : Row(category='vehicle', nb_lignes=19, nb_paniers=19, montant_total=1682064.05, montant_moyen=88529.69)


Top 3                : [Row(category='vehicle', nb_lignes=19, nb_paniers=19, montant_total=1682064.05, montant_moyen=88529.69), Row(category='motorcycle', nb_lignes=23, nb_paniers=22, montant_total=609460.61, montant_moyen=26498.29), Row(category='mens-watches', nb_lignes=26, nb_paniers=24, montant_total=593063.86, montant_moyen=22810.15)]


+-------+-----+
|  gamme|count|
+-------+-----+
|  basse|  418|
|  haute|  195|
|moyenne|  187|
+-------+-----+



**Transformations** : `select`, `filter`, `withColumn`, `groupBy`, `orderBy`. Elles créent un nouveau DataFrame mais ne calculent rien.

**Actions** : `show`, `count`, `first`, `take`. Elles demandent un résultat, donc Spark lance vraiment le calcul.

## 4. Lazy evaluation

On écrit plusieurs transformations sans action, on regarde le nombre de jobs dans Spark UI, puis on lance une action et on regarde à nouveau.

In [10]:
jobs_avant = len(spark.sparkContext.statusTracker().getJobIdsForGroup())

plan_paresseux = (
    paiements
    .filter(F.col("status") == "PAID")
    .withColumn("mois", F.date_format("attempted_at", "yyyy-MM"))
    .withColumn("amount_eur_estime", F.round(F.col("amount") * 0.92, 2))
    .groupBy("mois")
    .agg(F.round(F.sum("amount"), 2).alias("encaisse"))
    .orderBy("mois")
)

jobs_apres_transfo = len(spark.sparkContext.statusTracker().getJobIdsForGroup())
print("Jobs avant les transformations :", jobs_avant)
print("Jobs après les transformations :", jobs_apres_transfo, "donc aucun nouveau job")
plan_paresseux.explain()

Jobs avant les transformations : 20
Jobs après les transformations : 20 donc aucun nouveau job
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [mois#601 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(mois#601 ASC NULLS FIRST, 8), ENSURE_REQUIREMENTS, [plan_id=579]
      +- HashAggregate(keys=[mois#601], functions=[sum(amount#18)])
         +- Exchange hashpartitioning(mois#601, 8), ENSURE_REQUIREMENTS, [plan_id=576]
            +- HashAggregate(keys=[mois#601], functions=[partial_sum(amount#18)])
               +- Project [amount#18, date_format(attempted_at#19, yyyy-MM, Some(Europe/Paris)) AS mois#601]
                  +- BatchScan MongoTable()[amount#18, attempted_at#19] MongoScan{namespaceDescription=commerce.raw_payments} RuntimeFilters: []




In [11]:
plan_paresseux.show()
jobs_apres_action = len(spark.sparkContext.statusTracker().getJobIdsForGroup())
print("Jobs après l'action show() :", jobs_apres_action, f"donc {jobs_apres_action - jobs_apres_transfo} nouveau(x) job(s)")

+-------+---------+
|   mois| encaisse|
+-------+---------+
|2025-01|940419.71|
|2025-02|116920.24|
|2025-03|456418.27|
|2025-04|134104.03|
|2025-05|256691.85|
|2025-06|194348.97|
|2025-07| 43706.81|
|2025-08| 31699.65|
|2025-09|127247.21|
|2025-10| 94509.37|
+-------+---------+

Jobs après l'action show() : 22 donc 2 nouveau(x) job(s)


**Ce qu'on voit :** avant et après les transformations il y a le même nombre de jobs (20). Spark a juste préparé un plan, visible avec `explain()`. Dès qu'on fait `show()`, on passe à 22 jobs. C'est l'action qui déclenche le calcul.

## 5. Partitions et Spark UI

In [12]:
print("Cœurs utilisés (defaultParallelism) :", spark.sparkContext.defaultParallelism)
print("Partitions de commandes               :", commandes.rdd.getNumPartitions())
print("Partitions de paiements               :", paiements.rdd.getNumPartitions())
commandes.groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition").show()

Cœurs utilisés (defaultParallelism) : 8
Partitions de commandes               : 1
Partitions de paiements               : 1


+---------+-----+
|partition|count|
+---------+-----+
|        0|  800|
+---------+-----+



In [13]:
commandes_4 = commandes.repartition(4)
print("Après repartition(4) :", commandes_4.rdd.getNumPartitions(), "partitions")
commandes_4.groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition").show()
commandes_4.groupBy("category").agg(F.sum("discounted_line_total")).count()

Après repartition(4) : 4 partitions


+---------+-----+
|partition|count|
+---------+-----+
|        0|  200|
|        1|  200|
|        2|  200|
|        3|  200|
+---------+-----+



24

**Ce qu'on voit :** les 800 lignes sont dans 1 seule partition, donc chaque étape n'a qu'une task. Après `repartition(4)` on a 4 partitions de 200 lignes et Spark UI montre 4 tasks.

Une action lance un **job**. Un job est coupé en **stages**. Chaque stage lance une **task par partition**. Plus il y a de partitions, plus Spark peut travailler en parallèle.

## 6. Indicateurs BigData Commerce Analytics

Le chiffre d'affaires par catégorie se calcule avec `discounted_line_total` ligne par ligne. Pour l'argent encaissé, on ne garde que les paiements PAID.

In [14]:
ca_categorie = (
    commandes
    .groupBy("category")
    .agg(F.round(F.sum("discounted_line_total"), 2).alias("chiffre_affaires"), F.countDistinct("cart_id").alias("nb_paniers"))
    .withColumn("part_%", F.round(100 * F.col("chiffre_affaires") / F.sum("chiffre_affaires").over(Window.partitionBy()), 1))
    .orderBy(F.desc("chiffre_affaires"))
)
ca_categorie.show(50, truncate=False)

26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+----------------+----------+------+
|category           |chiffre_affaires|nb_paniers|part_%|
+-------------------+----------------+----------+------+
|vehicle            |1682064.05      |19        |48.7  |
|motorcycle         |609460.61       |22        |17.6  |
|mens-watches       |593063.86       |24        |17.2  |
|womens-watches     |249189.51       |17        |7.2   |
|laptops            |85006.24        |16        |2.5   |
|furniture          |77674.97        |21        |2.2   |
|smartphones        |73130.95        |52        |2.1   |
|mobile-accessories |26160.99        |61        |0.8   |
|tablets            |14890.16        |13        |0.4   |
|mens-shoes         |8278.9          |22        |0.2   |
|womens-bags        |5849.91         |20        |0.2   |
|kitchen-accessories|5714.33         |94        |0.2   |
|fragrances         |4958.05         |22        |0.1   |
|womens-dresses     |4064.13         |19        |0.1   |
|sports-accessories |3776.68   

In [15]:
payes = paiements.filter(F.col("status") == "PAID")

top_clients = (
    payes
    .groupBy("user_id")
    .agg(F.round(F.sum("amount"), 2).alias("montant_encaisse"), F.count("*").alias("nb_paiements"))
    .orderBy(F.desc("montant_encaisse"))
    .limit(10)
)
top_clients.show()

+-------+----------------+------------+
|user_id|montant_encaisse|nb_paiements|
+-------+----------------+------------+
|     30|       155507.16|           1|
|     52|       130175.28|           1|
|     95|       129783.31|           1|
|     33|       125461.58|           1|
|     18|       124978.81|           1|
|    155|       110543.89|           1|
|     78|       109890.08|           1|
|     27|       100496.36|           1|
|    123|        78514.37|           1|
|     71|        72541.01|           1|
+-------+----------------+------------+



In [16]:
taux_refus = paiements.agg(
    F.count("*").alias("tentatives"),
    F.sum(F.when(F.col("status") == "REFUSED", 1).otherwise(0)).alias("refus"),
    F.sum(F.when(F.col("status") == "PENDING", 1).otherwise(0)).alias("en_attente"),
    F.sum(F.when(F.col("status") == "PAID", 1).otherwise(0)).alias("acceptes"),
).withColumn("taux_refus_%", F.round(100 * F.col("refus") / F.col("tentatives"), 1))
taux_refus.show()

+----------+-----+----------+--------+------------+
|tentatives|refus|en_attente|acceptes|taux_refus_%|
+----------+-----+----------+--------+------------+
|       208|   43|        17|     148|        20.7|
+----------+-----+----------+--------+------------+



In [17]:
evolution_mensuelle = (
    payes
    .withColumn("mois", F.date_format("attempted_at", "yyyy-MM"))
    .groupBy("mois")
    .agg(F.round(F.sum("amount"), 2).alias("encaisse"), F.count("*").alias("nb_paiements"))
    .orderBy("mois")
)
evolution_mensuelle.show(24)

+-------+---------+------------+
|   mois| encaisse|nb_paiements|
+-------+---------+------------+
|2025-01|940419.71|          40|
|2025-02|116920.24|          20|
|2025-03|456418.27|          26|
|2025-04|134104.03|          14|
|2025-05|256691.85|          19|
|2025-06|194348.97|          12|
|2025-07| 43706.81|           4|
|2025-08| 31699.65|           4|
|2025-09|127247.21|           5|
|2025-10| 94509.37|           4|
+-------+---------+------------+



In [18]:
repartition_statuts = (
    paiements.groupBy("status")
    .agg(F.count("*").alias("nb"), F.round(F.sum("amount"), 2).alias("montant"))
    .orderBy(F.desc("nb"))
)
repartition_statuts.show()

+-------+---+----------+
| status| nb|   montant|
+-------+---+----------+
|   PAID|148|2396066.11|
|REFUSED| 43| 804575.17|
|PENDING| 17|  256068.3|
+-------+---+----------+



### Les chiffres à retenir

In [19]:
ca_pd = ca_categorie.toPandas()
top_pd = top_clients.toPandas()
refus_pd = taux_refus.toPandas().iloc[0]
mois_pd = evolution_mensuelle.toPandas()
statuts_pd = repartition_statuts.toPandas().set_index("status")

ca_total = ca_pd["chiffre_affaires"].sum()
encaisse = statuts_pd.loc["PAID", "montant"] if "PAID" in statuts_pd.index else 0
top3 = ca_pd.head(3)
part_top3 = 100 * top3["chiffre_affaires"].sum() / ca_total
part_top10_clients = 100 * top_pd["montant_encaisse"].sum() / encaisse if encaisse else 0
meilleur_mois = mois_pd.loc[mois_pd["encaisse"].idxmax()] if len(mois_pd) else None

print(f"Chiffre d'affaires des lignes (toutes tentatives) : {ca_total:,.2f} USD sur {len(ca_pd)} catégories.")
print(f"Catégories en tête : {', '.join(top3[ca_pd.columns[0]].astype(str))} ({part_top3:.1f} % du CA à elles trois).")
print(f"Montant réellement encaissé (PAID) : {encaisse:,.2f} USD, soit {100 * encaisse / ca_total:.1f} % du montant tenté.")
print(f"Taux de refus : {refus_pd['taux_refus_%']} % ({int(refus_pd['refus'])} refus sur {int(refus_pd['tentatives'])} tentatives), {int(refus_pd['en_attente'])} paiements en attente.")
print(f"Les 10 meilleurs clients représentent {part_top10_clients:.1f} % du montant encaissé ; le premier (user_id {int(top_pd.iloc[0]['user_id'])}) a payé {top_pd.iloc[0]['montant_encaisse']:,.2f} USD.")
if meilleur_mois is not None:
    print(f"Meilleur mois : {meilleur_mois['mois']} avec {meilleur_mois['encaisse']:,.2f} USD encaissés ({len(mois_pd)} mois couverts).")

26/11/26 11:58:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 11:58:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/26 1

Chiffre d'affaires des lignes (toutes tentatives) : 3,456,709.58 USD sur 24 catégories.
Catégories en tête : vehicle, motorcycle, mens-watches (83.4 % du CA à elles trois).
Montant réellement encaissé (PAID) : 2,396,066.11 USD, soit 69.3 % du montant tenté.
Taux de refus : 20.7 % (43 refus sur 208 tentatives), 17 paiements en attente.
Les 10 meilleurs clients représentent 47.5 % du montant encaissé ; le premier (user_id 30) a payé 155,507.16 USD.
Meilleur mois : 2025-01 avec 940,419.71 USD encaissés (10 mois couverts).


### Commentaire des résultats

**Chiffre d'affaires par catégorie.** Sur 24 catégories, le chiffre d'affaires des lignes (après remise) est de 3 456 709,58 USD. Trois catégories font presque tout : les véhicules (48,7 %), les motos (17,6 %) et les montres homme (17,2 %), soit 83,4 % à elles trois. Ce sont des produits très chers achetés en petite quantité : 19 paniers seulement contiennent un véhicule. À l'inverse, les accessoires de cuisine et l'alimentation sont dans 94 paniers chacun mais pèsent 0,2 % et 0,1 % du chiffre d'affaires. Le nombre de ventes et l'argent rapporté ne racontent donc pas la même chose.

**Top 10 clients.** Les 10 meilleurs clients font 47,5 % de l'argent encaissé, presque la moitié. Le premier (user_id 30) a payé 155 507,16 USD. Chaque client n'a qu'un seul paiement : ce sont donc les clients qui ont un véhicule ou une moto dans leur panier qui arrivent en tête, pas des clients fidèles.

**Taux de refus.** 43 paiements refusés sur 208 tentatives, soit 20,7 %, très proche des 20 % prévus par la simulation. Il y a aussi 17 paiements en attente (8,2 %) et 148 acceptés (71,2 %).

**Évolution mensuelle.** Janvier 2025 arrive très loin devant avec 940 419,71 USD encaissés et 40 paiements, puis ça baisse et il n'y a plus rien après octobre. Ce n'est pas une vraie tendance : la date d'un paiement est la plus ancienne date des lignes du panier, et comme ces dates sont tirées au hasard par le script, la plus ancienne tombe souvent en début d'année. Il ne faut pas en tirer de conclusion sur la saison des ventes.

**Encaissé et non encaissé.** Sur 3 456 709,58 USD tentés, seulement 2 396 066,11 USD sont vraiment encaissés (69,3 %). Les 1 060 643,47 USD restants sont refusés (804 575,17) ou en attente (256 068,30). Si on comptait tout, on surestimerait le chiffre d'affaires d'environ 44 %.

Toutes ces valeurs viennent de données de démonstration DummyJSON et d'une simulation : elles servent à apprendre la méthode, pas à décrire une vraie boutique.

## Analyse courte

**Répartition des paiements.** On a simulé un paiement par panier, donc 208 paiements. Résultat : 148 PAID (71,2 %, prévu 70 %), 43 REFUSED (20,7 %, prévu 20 %) et 17 PENDING (8,2 %, prévu 10 %). Les écarts sont normaux : avec seulement 208 tirages au hasard, on ne tombe jamais pile sur les pourcentages. Les contrôles du notebook 01 montrent qu'aucun panier n'a été payé deux fois et que MongoDB refuse un doublon grâce à l'index unique.

**Lazy evaluation.** Avant de définir les transformations, Spark avait lancé 20 jobs. Après avoir écrit filter, withColumn, groupBy et orderBy, il y en avait toujours 20 : rien n'a été calculé, Spark a seulement préparé un plan (visible avec explain). Dès l'action show(), on est passé à 22 jobs. C'est l'action qui déclenche vraiment le calcul.

**Partitions et Spark UI.** Les 800 commandes tiennent dans 1 seule partition, donc chaque stage n'a qu'une task (on voit 1/1 dans l'onglet Jobs). Après repartition(4), les données sont coupées en 4 partitions de 200 lignes, et Spark UI montre 4/4 tasks : une task par partition. Les jobs marqués skipped sont des étapes que Spark n'a pas refaites car il avait déjà le résultat. Avec si peu de données les temps sont très courts (entre 15 ms et 1 s) et ne représentent pas un vrai cluster.

### Sauvegarde pour le jour 3

In [20]:
from pathlib import Path

dossier = Path("resultats")
dossier.mkdir(exist_ok=True)
indicateurs = {
    "ca_par_categorie": ca_pd,
    "top10_clients": top_pd,
    "taux_refus": taux_refus.toPandas(),
    "evolution_mensuelle": mois_pd,
    "repartition_statuts": statuts_pd.reset_index(),
}
for nom, table in indicateurs.items():
    table.to_csv(dossier / f"{nom}.csv", index=False)
    spark.createDataFrame(table).write.format("mongodb").mode("overwrite").option("database", DB).option("collection", f"ind_{nom}").save()
    print(f"resultats/{nom}.csv et commerce.ind_{nom} enregistrés")

resultats/ca_par_categorie.csv et commerce.ind_ca_par_categorie enregistrés


resultats/top10_clients.csv et commerce.ind_top10_clients enregistrés


resultats/taux_refus.csv et commerce.ind_taux_refus enregistrés


resultats/evolution_mensuelle.csv et commerce.ind_evolution_mensuelle enregistrés


resultats/repartition_statuts.csv et commerce.ind_repartition_statuts enregistrés


## Questions de synthèse

**Collection MongoDB ou DataFrame Spark ?** Une collection MongoDB stocke des documents JSON sur le disque, et ils peuvent avoir des champs différents. Un DataFrame Spark sert à calculer : c'est un tableau avec des colonnes fixes, découpé en partitions, qui disparaît quand on arrête Spark.

**Transformation ou action ?** Une transformation prépare un calcul sans le faire. Une action demande un résultat et lance le calcul.

**Pourquoi lazy evaluation ?** Spark attend la première action pour tout calculer d'un coup. Comme il connaît toute la chaîne, il peut l'optimiser, par exemple en filtrant le plus tôt possible.

**Partitions, jobs, stages, tasks ?** Les données sont coupées en partitions. Une action crée un job, le job est coupé en stages, et chaque stage lance une task par partition.

**Pourquoi le CA encaissé seulement avec les PAID ?** Parce que seuls les PAID sont de l'argent vraiment reçu. Un REFUSED n'a rien rapporté et un PENDING peut encore échouer. Si on les comptait, le chiffre d'affaires serait trop haut de 44 %.